In [9]:
import pandas as pd
import os

In [10]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

df = pd.read_csv('/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/MainDataFrame_Lineages_Augmin.csv')

In [11]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250724", "position": 2, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 3, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 4, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 5, "target": "LUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 6, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 7, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 8, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 9, "target": "HAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_si{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

In [12]:
df["Frame"] = df.Frame.astype(int) + 1 #otherwise the trackmate and mitotic timing frames are not aligned

df = df.merge(df_onsets, how = "outer")

df.head()
    

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Cell_Cycle_hours,Seen_sister,Random_ID,Seen_granny,,Label,X,Y,Frame_onset,dataset
0,0,0,2005,True,2005,666.091036,713.323699,118,2,20250724,...,NaN,NaN,2934,NaN,3,tomeasure:2005:109,662.915,696.060,109,20250724
1,2,0,2027,True,2005.2027,662.224034,670.786681,534,2,20250724,...,48.533333,NaN,2959,NaN,26,tomeasure:2027:523,604.081,651.314,523,20250724
2,4,1,2096,True,2096,518.040115,698.960550,211,2,20250724,...,NaN,NaN,2423,NaN,11,tomeasure:2096:201,528.674,692.746,201,20250724
3,6,2,2121,True,2121,499.809964,637.640952,119,2,20250724,...,NaN,NaN,2096,NaN,4,tomeasure:2121:108,515.416,613.196,108,20250724
4,8,3,2153,True,2153,874.632911,867.175122,4,2,20250724,...,NaN,NaN,2529,NaN,15,tomeasure:2153:1,869.247,863.446,1,20250724


In [13]:
df["Frames_in_mitosis"] = (df.Frame.astype(int) - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

In [14]:
def getInterphaseDuration(x, dataframe = df):
    # should be the same as cell_cycle - mitotic duration
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.dataset == x.dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None


def getMetaphaseArrest(x):
    if x > 100:
        return "> 100 min"
   # elif x > 45:
   #     return "045-90"
    else:
        return "< 100 min"

def getMetaphaseArrest_mother(x):
    if x > 0:
        if x > 100:
            return "> 100 min"
       # elif x > 45:
       #     return "045-90"
        else:
            return "< 100 min"
    else:
        pass

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * 7 / 60 # Augmin depletion time at NEB
df["Frame_hrs"] = df.Frame * 7 / 60 # Augmin depletion time at anaphase

# Create Bins (one bin for every 12 h of depletion, binning cells with the same onset of mitotis)
interval_range_depletion = pd.interval_range(start = 0, freq = 12, end = 96) 
df['Augmin_depletion_bin'] = pd.cut(df['Frame_onset_hrs'], bins = interval_range_depletion).astype(str)

df.head()

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Mitotic_duration_mins,Interphase_duration_mins,Interphase_duration_hrs,Mitotic_duration_hrs,Mother_Mitotic_duration_mins,Mother_arrested,Metaphase_arrested,Frame_onset_hrs,Frame_hrs,Augmin_depletion_bin
0,0,0,2005,True,2005,666.091036,713.323699,118,2,20250724,...,63,NaN,NaN,1.050000,NaN,None,< 100 min,12.716667,13.766667,"(12, 24]"
1,2,0,2027,True,2005.2027,662.224034,670.786681,534,2,20250724,...,77,2835.0,47.25,1.283333,63.0,< 100 min,< 100 min,61.016667,62.300000,"(60, 72]"
2,4,1,2096,True,2096,518.040115,698.960550,211,2,20250724,...,70,NaN,NaN,1.166667,NaN,None,< 100 min,23.450000,24.616667,"(12, 24]"
3,6,2,2121,True,2121,499.809964,637.640952,119,2,20250724,...,77,NaN,NaN,1.283333,NaN,None,< 100 min,12.600000,13.883333,"(12, 24]"
4,8,3,2153,True,2153,874.632911,867.175122,4,2,20250724,...,21,NaN,NaN,0.350000,NaN,None,< 100 min,0.116667,0.466667,"(0, 12]"


In [15]:
def getCondition(x):
    if x < 6:
        return "siLUC1"
    else:
        return "siHAUS6"

df["Condition"] = df.Position.apply(getCondition)

In [16]:
#selected_df = df[["Dataset", "Source_ID", "Position", "Frame", "Frame_onset", "X", "Y"]]

#selected_df.to_csv(outputFolder + "SelectedDataFrame_MitoticStopwatch_Augmin.csv")

In [17]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Augmin.csv")

print("Finished.")

Finished.
